In [1]:
import torch
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")

cuda: True
device: Tesla T4


In [2]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/0_potato_project_v1/scripts")
import config, data, torch

data.setup_data()
df = data.load_manifest()
print("cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")
print("manifest:", df.shape)

restoring /content/drive/MyDrive/0_potato_project_v1/data/potato_raw/raw -> /content/potato
  Early_blight -> Potato___Early_blight
  Late_blight -> Potato___Late_blight
  Healthy -> Potato___healthy
cuda: True | Tesla T4
manifest: (2152, 13)


In [4]:

# 04 : STEP 1: MobileNetV3-Large with a 3-class head
import sys
sys.path.insert(0, "/content/drive/MyDrive/0_potato_project_v1/scripts")

import torch
import torch.nn as nn
from torchvision import models

import config as C


def build_model(num_classes=C.NUM_CLASSES, pretrained=True, dropout=0.2):
    """MobileNetV3-Large, ImageNet weights, final layer resized to num_classes."""
    weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.mobilenet_v3_large(weights=weights)

    in_f = m.classifier[3].in_features          # 1280
    m.classifier[3] = nn.Linear(in_f, num_classes)
    m.classifier[2].p = dropout                 # explicit, not inherited silently
    return m


#  verify
model = build_model()
total = sum(p.numel() for p in model.parameters())
clf   = sum(p.numel() for p in model.classifier.parameters())

print(f"feature blocks : {len(model.features)}")
print(f"params total   : {total:,}")
print(f"params clf     : {clf:,}")
print(f"final layer    : {model.classifier[3]}")
print(f"dropout p      : {model.classifier[2].p}")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 150MB/s]

feature blocks : 17
params total   : 4,205,875
params clf     : 1,233,923
final layer    : Linear(in_features=1280, out_features=3, bias=True)
dropout p      : 0.2


In [5]:

# 04 :STEP 2: stage control + BatchNorm handling
import types

UNFREEZE_FROM = 12          # stage 2 unfreezes features[12:] + classifier


def _count(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    fz = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return tr, fz


def set_stage(model, stage, unfreeze_from=UNFREEZE_FROM):
    """stage 1 = head only (backbone frozen). stage 2 = head + late blocks."""
    if stage not in (1, 2):
        raise ValueError("stage must be 1 or 2")

    for p in model.features.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

    if stage == 2:
        for blk in model.features[unfreeze_from:]:
            for p in blk.parameters():
                p.requires_grad = True

    model._stage = stage
    _install_bn_guard(model)
    return model


def _install_bn_guard(model):
    """Override .train() so frozen BatchNorm layers never update running stats."""
    if getattr(model, "_bn_guard", False):
        return model

    original_train = model.train

    def train(self, mode=True):
        original_train(mode)
        if mode:
            for i, blk in enumerate(self.features):
                if i < UNFREEZE_FROM or self._stage == 1:
                    for m in blk.modules():
                        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                            m.eval()
        return self

    model.train = types.MethodType(train, model)
    model._bn_guard = True
    return model


# verify
model = build_model()

for stage in (1, 2):
    set_stage(model, stage)
    model.train()
    tr, fz = _count(model)
    bn_train = sum(1 for m in model.features.modules()
                   if isinstance(m, torch.nn.modules.batchnorm._BatchNorm) and m.training)
    bn_total = sum(1 for m in model.features.modules()
                   if isinstance(m, torch.nn.modules.batchnorm._BatchNorm))
    print(f"stage {stage}: trainable {tr:>9,}  frozen {fz:>9,}  "
          f"BN in train mode {bn_train}/{bn_total}")

# stats must not move when the backbone is frozen
set_stage(model, 1); model.train()
bn = [m for m in model.features.modules()
      if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)][0]
before = bn.running_mean.clone()
model(torch.randn(4, 3, C.IMG_SIZE, C.IMG_SIZE))
print("stage 1 running stats unchanged:", torch.equal(before, bn.running_mean))
print("forward output shape:", tuple(model(torch.randn(4, 3, 224, 224)).shape))

stage 1: trainable 1,233,923  frozen 2,971,952  BN in train mode 0/46
stage 2: trainable 3,799,507  frozen   406,368  BN in train mode 13/46
stage 1 running stats unchanged: True
forward output shape: (4, 3)


In [6]:
# 04: STEP 3: optimizer param groups + checkpoint save/load
from pathlib import Path


def param_groups(model, stage, lr_head=C.LR_HEAD, lr_backbone=C.LR_BACKBONE):
    """Trainable params only, split so stage 2 moves the backbone slower."""
    head = [p for p in model.classifier.parameters() if p.requires_grad]
    back = [p for p in model.features.parameters() if p.requires_grad]

    groups = [{"params": head, "lr": lr_head if stage == 1 else lr_head / 10,
               "name": "head"}]
    if back:
        groups.append({"params": back, "lr": lr_backbone, "name": "backbone"})
    return groups


def save_checkpoint(path, model, *, fold, stage, epoch, metrics, aug_mode=None):
    """Weights plus everything needed to reload and serve them correctly."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict":    model.state_dict(),
        "arch":          "mobilenet_v3_large",
        "class_to_idx":  C.CLASS_TO_IDX,
        "img_size":      C.IMG_SIZE,
        "aug_mode":      aug_mode or C.AUG_MODE,
        "unfreeze_from": UNFREEZE_FROM,
        "fold":          fold,
        "stage":         stage,
        "epoch":         epoch,
        "metrics":       metrics,
        "seed":          C.SEED,
    }, path)
    return path


def load_checkpoint(path, device=None):
    """Rebuild the model (no download) and restore weights. Returns (model, meta)."""
    ck = torch.load(path, map_location=device or "cpu", weights_only=False)
    if ck["class_to_idx"] != C.CLASS_TO_IDX:
        raise ValueError(f"class mapping mismatch: {ck['class_to_idx']}")

    m = build_model(num_classes=len(ck["class_to_idx"]), pretrained=False)
    m.load_state_dict(ck["state_dict"])
    m.eval()
    if device:
        m.to(device)
    return m, {k: v for k, v in ck.items() if k != "state_dict"}


# ── verify
model = build_model()

for stage in (1, 2):
    set_stage(model, stage)
    g = param_groups(model, stage)
    print(f"stage {stage}: " + "  ".join(
        f"{x['name']} {sum(p.numel() for p in x['params']):,} @ {x['lr']:g}" for x in g))

tmp = save_checkpoint("/content/_tmp.pt", model, fold=0, stage=2, epoch=1,
                      metrics={"macro_f1": 0.0})
m2, meta = load_checkpoint(tmp, device="cuda")

a = model.classifier[3].weight.detach().cpu()
b = m2.classifier[3].weight.detach().cpu()
print("weights round-trip identical:", torch.equal(a, b))
print("meta:", {k: meta[k] for k in ("arch", "img_size", "aug_mode", "unfreeze_from", "fold")})
print("reloaded device:", next(m2.parameters()).device, "| training mode:", m2.training)
Path(tmp).unlink()

stage 1: head 1,233,923 @ 0.001
stage 2: head 1,233,923 @ 0.0001  backbone 2,565,584 @ 1e-05
weights round-trip identical: True
meta: {'arch': 'mobilenet_v3_large', 'img_size': 224, 'aug_mode': 'baseline', 'unfreeze_from': 12, 'fold': 0}
reloaded device: cuda:0 | training mode: False


In [7]:
%%writefile /content/drive/MyDrive/0_potato_project_v1/scripts/model.py
"""MobileNetV3-Large builder, two-stage freeze control, checkpoint I/O.

Stage 1 trains the head on a frozen backbone; stage 2 additionally unfreezes
features[UNFREEZE_FROM:]. BatchNorm running statistics are held fixed wherever
the surrounding weights are frozen -- see _install_bn_guard.
"""
import types
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models

import config as C

UNFREEZE_FROM = 12          # stage 2 unfreezes features[12:] + classifier


# ── Build ─────────────────────────────────────────────────────────────────
def build_model(num_classes=C.NUM_CLASSES, pretrained=True, dropout=0.2):
    """ImageNet-pretrained MobileNetV3-Large with the final layer resized.

    Only classifier[3] is replaced; the 960->1280 projection is pretrained
    feature machinery, not an ImageNet label mapping, and is kept.
    """
    weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.mobilenet_v3_large(weights=weights)

    in_f = m.classifier[3].in_features          # 1280
    m.classifier[3] = nn.Linear(in_f, num_classes)
    m.classifier[2].p = dropout
    return m


# ── Stage control ─────────────────────────────────────────────────────────
def set_stage(model, stage, unfreeze_from=UNFREEZE_FROM):
    """stage 1 = head only (backbone frozen). stage 2 = head + late blocks."""
    if stage not in (1, 2):
        raise ValueError("stage must be 1 or 2")

    for p in model.features.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

    if stage == 2:
        for blk in model.features[unfreeze_from:]:
            for p in blk.parameters():
                p.requires_grad = True

    model._stage = stage
    _install_bn_guard(model)
    return model


def _install_bn_guard(model):
    """Override .train() so frozen BatchNorm layers never update running stats.

    requires_grad=False stops gradients but NOT the running mean/var update,
    which happens on every forward pass in training mode. Left unhandled, a
    'frozen' backbone drifts its normalisation statistics away from the weights
    that were tuned against them -- silent, and it only shows up as validation
    metrics that are inexplicably worse than training.

    Wired into .train() rather than the training loop because .train() is called
    every epoch and this must hold every time.
    """
    if getattr(model, "_bn_guard", False):
        return model

    original_train = model.train

    def train(self, mode=True):
        original_train(mode)
        if mode:
            for i, blk in enumerate(self.features):
                if i < UNFREEZE_FROM or self._stage == 1:
                    for m in blk.modules():
                        if isinstance(m, nn.modules.batchnorm._BatchNorm):
                            m.eval()
        return self

    model.train = types.MethodType(train, model)
    model._bn_guard = True
    return model


def count_params(model):
    """(trainable, frozen) parameter counts."""
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    fz = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return tr, fz


# ── Optimizer groups ──────────────────────────────────────────────────────
def param_groups(model, stage, lr_head=C.LR_HEAD, lr_backbone=C.LR_BACKBONE):
    """Trainable params only, split so stage 2 moves the backbone slower.

    Passing frozen tensors to the optimizer allocates state for parameters that
    never update, so this filters on requires_grad.
    """
    head = [p for p in model.classifier.parameters() if p.requires_grad]
    back = [p for p in model.features.parameters() if p.requires_grad]

    groups = [{"params": head, "lr": lr_head if stage == 1 else lr_head / 10,
               "name": "head"}]
    if back:
        groups.append({"params": back, "lr": lr_backbone, "name": "backbone"})
    return groups


# ── Checkpoints ───────────────────────────────────────────────────────────
def save_checkpoint(path, model, *, fold, stage, epoch, metrics, aug_mode=None):
    """Weights plus everything needed to reload and serve them correctly.

    class_to_idx and aug_mode live inside the file deliberately: the serving
    API must not depend on a separate mapping that can drift out of sync.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict":    model.state_dict(),
        "arch":          "mobilenet_v3_large",
        "class_to_idx":  C.CLASS_TO_IDX,
        "img_size":      C.IMG_SIZE,
        "aug_mode":      aug_mode or C.AUG_MODE,
        "unfreeze_from": UNFREEZE_FROM,
        "fold":          fold,
        "stage":         stage,
        "epoch":         epoch,
        "metrics":       metrics,
        "seed":          C.SEED,
    }, path)
    return path


def load_checkpoint(path, device=None):
    """Rebuild the model (no weight download) and restore. Returns (model, meta)."""
    ck = torch.load(path, map_location=device or "cpu", weights_only=False)
    if ck["class_to_idx"] != C.CLASS_TO_IDX:
        raise ValueError(f"class mapping mismatch: {ck['class_to_idx']}")

    m = build_model(num_classes=len(ck["class_to_idx"]), pretrained=False)
    m.load_state_dict(ck["state_dict"])
    m.eval()
    if device:
        m.to(device)
    return m, {k: v for k, v in ck.items() if k != "state_dict"}

Writing /content/drive/MyDrive/0_potato_project_v1/scripts/model.py
